# <span style='color:steelblue'>Overview</span>

Human Activity Recognition (HAR) from smartphone inertial sensors.

**Dataset**: UCI HAR — 30 subjects, 6 activities, 9-channel raw signals (128 timesteps).

**Models**:
- Classical: RandomForest on 561 hand-engineered features
- Deep: LSTM, GRU, 1D-CNN — all trained on raw 9×128 signals

**Evaluation**: official subject-independent train/test split. Metrics: accuracy, macro-F1, per-class F1, confusion matrix.

In [ ]:
import sys, os, json, logging, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 100})
sns.set_style('whitegrid')
sns.set_palette('muted')

# Reuse har.py functions
sys.path.insert(0, '.')
from har import *

# <span style='color:steelblue'>Data Loading & EDA</span>

Auto-download, load raw signals and engineered features, visualise example sequences per activity.

In [ ]:
data_dir = Path('./data')
root = download_and_extract(data_dir)
X_train_seq, y_train = load_raw_signals(root, 'train')
X_test_seq, y_test = load_raw_signals(root, 'test')
X_train_eng, _ = load_engineered_features(root, 'train')
X_test_eng, _ = load_engineered_features(root, 'test')

print(f'Train raw signals : {X_train_seq.shape}')
print(f'Test  raw signals : {X_test_seq.shape}')
print(f'Train eng features: {X_train_eng.shape}')
print(f'Test  eng features: {X_test_eng.shape}')

In [ ]:
# Class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for split, y, ax in [('Train', y_train, ax1), ('Test', y_test, ax2)]:
    unique, counts = np.unique(y, return_counts=True)
    labels = [ACTIVITIES[i] for i in unique]
    ax.bar(labels, counts, color=sns.color_palette('muted', 6))
    ax.set_title(f'{split} — Class Distribution')
    ax.set_ylabel('Samples')
    ax.tick_params(axis='x', rotation=30)
fig.tight_layout()
plt.show()

In [ ]:
# Plot example raw signal sequences (body-acc-x) per activity
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.ravel()
for act_idx in range(N_CLASSES):
    idxs = np.where(y_train == act_idx)[0]
    sample_idx = np.random.choice(idxs)
    ax = axes[act_idx]
    for ch in range(3):  # body acc x,y,z
        ax.plot(X_train_seq[sample_idx, ch], alpha=0.7, label=['X','Y','Z'][ch])
    ax.set_title(ACTIVITIES[act_idx])
    ax.set_xlabel('Timestep')
    ax.set_ylabel('Acceleration')
    ax.legend(fontsize=7)
fig.suptitle('Example Body-Acceleration Sequences per Activity', fontsize=14)
fig.tight_layout()
plt.show()

# <span style='color:steelblue'>Classical Baseline — RandomForest</span>

Train on the 561 hand-crafted features (mean, std, entropy, correlation, etc.). Strong baseline due to expert feature engineering.

In [ ]:
set_seed(SEED)
rf_result = train_random_forest(X_train_eng, y_train, X_test_eng, y_test)
print(f"RF  accuracy={rf_result['accuracy']:.4f}  macro-F1={rf_result['macro_f1']:.4f}")
print(f"Per-class F1: {json.dumps(rf_result['per_class_f1'], indent=2)}")

# <span style='color:steelblue'>LSTM</span>

2-layer bidirectional LSTM (hidden=128) trained on 9×128 raw signals.

In [ ]:
set_seed(SEED)
lstm_result = train_deep_model('lstm', X_train_seq, y_train, X_test_seq, y_test,
                               out_dir=Path('./out'), epochs=20, lr=1e-3, batch_size=64)

In [ ]:
# LSTM training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = lstm_result['history']
ax1.plot(h['train_loss'], label='Train')
ax1.plot(h['test_loss'], label='Test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('LSTM — Loss Curve'); ax1.legend()
ax2.plot(h['test_acc'], color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('LSTM — Test Accuracy');
fig.tight_layout(); plt.show()

# <span style='color:steelblue'>GRU</span>

2-layer bidirectional GRU (hidden=128), same architecture otherwise.

In [ ]:
set_seed(SEED)
gru_result = train_deep_model('gru', X_train_seq, y_train, X_test_seq, y_test,
                              out_dir=Path('./out'), epochs=20, lr=1e-3, batch_size=64)

In [ ]:
# GRU training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = gru_result['history']
ax1.plot(h['train_loss'], label='Train')
ax1.plot(h['test_loss'], label='Test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('GRU — Loss Curve'); ax1.legend()
ax2.plot(h['test_acc'], color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('GRU — Test Accuracy');
fig.tight_layout(); plt.show()

# <span style='color:steelblue'>1D-CNN</span>

3 conv blocks (kernel 9→7→5) over the time axis with GAP. Learns local temporal patterns without recurrence.

In [ ]:
set_seed(SEED)
cnn_result = train_deep_model('cnn1d', X_train_seq, y_train, X_test_seq, y_test,
                              out_dir=Path('./out'), epochs=20, lr=1e-3, batch_size=64)

In [ ]:
# CNN training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
h = cnn_result['history']
ax1.plot(h['train_loss'], label='Train')
ax1.plot(h['test_loss'], label='Test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('1D-CNN — Loss Curve'); ax1.legend()
ax2.plot(h['test_acc'], color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('1D-CNN — Test Accuracy');
fig.tight_layout(); plt.show()

# <span style='color:steelblue'>Comparison Table & Confusion Matrices</span>

Side-by-side comparison of all models: accuracy, macro-F1, per-class F1.

In [ ]:
all_results = [rf_result, lstm_result, gru_result, cnn_result]

# Summary table
import pandas as pd
rows = []
for r in all_results:
    row = {'Model': r['model'], 'Accuracy': r['accuracy'], 'Macro-F1': r['macro_f1']}
    row.update({f'F1({a[:4]})': r['per_class_f1'][a] for a in ACTIVITIES})
    rows.append(row)
df = pd.DataFrame(rows).set_index('Model')
display(df.style.background_gradient(cmap='Blues', axis=0).format('{:.4f}'))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, r in zip(axes, all_results):
    cm = np.array(r['confusion_matrix'])
    disp = ConfusionMatrixDisplay(cm, display_labels=ACTIVITIES)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
    ax.set_title(f"{r['model'].upper()}")
fig.tight_layout()
plt.show()

# <span style='color:steelblue'>Conclusion</span>

- The 1D-CNN typically converges fastest and matches or exceeds RNN variants on this task, since the 128-length sequences contain local motifs (stride patterns) that convolutions capture efficiently.
- LSTM and GRU perform comparably; bidirectional recurrence helps but adds parameters.
- The RandomForest on 561 engineered features remains a strong baseline — often exceeding or matching deep models because the feature set was designed by domain experts for exactly this task.
- All metrics are saved to `out/metrics.json`; best model weights are in `out/{model}_best.pt`.